**Table of contents**<a id='toc0_'></a>    
- [English-Vietnamese Data Preprocessing](#toc1_)    
  - [Prepare the notebook](#toc1_1_)    
  - [Read the raw dataset](#toc1_2_)    
  - [Define tokenizers](#toc1_3_)    
  - [Tokenize all splits](#toc1_4_)    
  - [Build the vocabularies](#toc1_5_)    
  - [Filter long or empty sentences](#toc1_6_)    
  - [Convert tokens to integer IDs](#toc1_7_)    
  - [Save the processed data](#toc1_8_)    
  - [Verify the saved vocabulary](#toc1_9_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[English-Vietnamese Data Preprocessing](#toc0_)

## <a id='toc1_1_'></a>[Prepare the notebook](#toc0_)

Import the project code and the libraries used for reading, tokenizing, and saving the dataset.


In [22]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import re
import pickle
from collections import Counter
from underthesea import word_tokenize as vi_tokenize
from src.vocab import Vocab

## <a id='toc1_2_'></a>[Read the raw dataset](#toc0_)

Load train, dev, and test sentence pairs from `data/raw/`.

The assertions make sure each English sentence has a matching Vietnamese sentence.


In [2]:
DATA_DIR = 'D:/AIResearcher/MachineTraslation/data/raw'

def read_lines(path):
    with open(path, encoding='utf-8') as f:
        return [line.strip() for line in f]

train_en = read_lines(f'{DATA_DIR}/train.en')
train_vi = read_lines(f'{DATA_DIR}/train.vi')
dev_en   = read_lines(f'{DATA_DIR}/tst2012.en')
dev_vi   = read_lines(f'{DATA_DIR}/tst2012.vi')
test_en  = read_lines(f'{DATA_DIR}/tst2013.en')
test_vi  = read_lines(f'{DATA_DIR}/tst2013.vi')

assert len(train_en) == len(train_vi)
assert len(dev_en) == len(dev_vi)
assert len(test_en) == len(test_vi)

print("Sample1: ")
print(f"Train English[0]: {train_en[0]}")
print(f"Train Vietnam[0]: {train_vi[0]}")

print("Sample2: ")
print(f"dev English[0]: {dev_en[0]}")
print(f"dev Vietnam[0]: {dev_vi[0]}")

print("Sample3: ")
print(f"test English[0]: {test_en[0]}")
print(f"test Vietnam[0]: {test_vi[0]}")

Sample1: 
Train English[0]: Rachel Pike : The science behind a climate headline
Train Vietnam[0]: Khoa học đằng sau một tiêu đề về khí hậu
Sample2: 
dev English[0]: How can I speak in 10 minutes about the bonds of women over three generations , about how the astonishing strength of those bonds took hold in the life of a four-year-old girl huddled with her young sister , her mother and her grandmother for five days and nights in a small boat in the China Sea more than 30 years ago , bonds that took hold in the life of that small girl and never let go -- that small girl now living in San Francisco and speaking to you today ?
dev Vietnam[0]: Làm sao tôi có thể trình bày trong 10 phút về sợi dây liên kết những người phụ nữ qua ba thế hệ , về việc làm thế nào những sợi dây mạnh mẽ đáng kinh ngạc ấy đã níu chặt lấy cuộc sống của một cô bé bốn tuổi co quắp với đứa em gái nhỏ của cô bé , với mẹ và bà trong suốt năm ngày đêm trên con thuyền nhỏ lênh đênh trên Biển Đông hơn 30 năm trước , những 

## <a id='toc1_3_'></a>[Define tokenizers](#toc0_)

Normalize English to lowercase and split punctuation into separate tokens. Vietnamese uses `underthesea` for word segmentation.


In [3]:
def tokenize_en(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([.,!?;:])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip().split()

def tokenize_vi(sentence):
    sentence = sentence.lower().strip()
    return vi_tokenize(sentence)

## <a id='toc1_4_'></a>[Tokenize all splits](#toc0_)

Apply the tokenizers to train, dev, and test. The vocabulary is still built only from the train tokens below.


In [4]:
from tqdm import tqdm

def tokenize_all(sentences, tokenize_fn):
    return [tokenize_fn(s) for s in tqdm(sentences)]

train_en_tok = tokenize_all(train_en, tokenize_en)
train_vi_tok = tokenize_all(train_vi, tokenize_vi)
dev_en_tok   = tokenize_all(dev_en, tokenize_en)
dev_vi_tok   = tokenize_all(dev_vi, tokenize_vi)
test_en_tok  = tokenize_all(test_en, tokenize_en)
test_vi_tok  = tokenize_all(test_vi, tokenize_vi)

100%|██████████| 1268/1268 [00:00<00:00, 1511.33it/s]


## <a id='toc1_5_'></a>[Build the vocabularies](#toc0_)

Build one English vocabulary and one Vietnamese vocabulary from the **training set only**.

The same vocabularies are reused for dev and test. Unseen words are mapped to `<unk>`.


In [15]:
print("Sample1: ")
print(f"Train English[0]: {train_en_tok[0]}")
print(f"Train Vietnam[0]: {train_vi_tok[0]}")

Sample1: 
Train English[0]: ['rachel', 'pike', ':', 'the', 'science', 'behind', 'a', 'climate', 'headline']
Train Vietnam[0]: ['khoa học', 'đằng', 'sau', 'một', 'tiêu đề', 'về', 'khí hậu']


## <a id='toc1_6_'></a>[Filter long or empty sentences](#toc0_)

Keep non-empty sentence pairs whose source and target lengths are at most `MAX_LEN`.


In [23]:
en_vocab = Vocab(train_en_tok)
vi_vocab = Vocab(train_vi_tok)

print(f"Vocab size [en_vocab]: {len(en_vocab)}")
print(f"Vocab size [vi_vocab]: {len(vi_vocab)}")

Vocab size [en_vocab]: 22276
Vocab size [vi_vocab]: 16254


In [ ]:
MAX_LEN = 50

def filter_pairs(src_tokens, tgt_tokens, max_len=MAX_LEN):
    pairs = [
        (src, tgt)
        for src, tgt in zip(src_tokens, tgt_tokens)
        if 0 < len(src) <= max_len
        and 0 < len(tgt) <= max_len
    ]

    print(f"Giữ lại {len(pairs)} / {len(src_tokens)} cặp câu")
    return pairs


train_pairs = filter_pairs(train_en_tok, train_vi_tok)
dev_pairs   = filter_pairs(dev_en_tok, dev_vi_tok)
test_pairs  = filter_pairs(test_en_tok, test_vi_tok)

Giữ lại 125798 / 133317 cặp câu
Giữ lại 1511 / 1553 cặp câu
Giữ lại 1192 / 1268 cặp câu


## <a id='toc1_7_'></a>[Convert tokens to integer IDs](#toc0_)

Replace each token with its vocabulary ID and add `<bos>` and `<eos>` around every sentence.


In [8]:
def numericalize_pairs(pairs, src_vocab, tgt_vocab):
    return [(src_vocab.encode(EN), tgt_vocab.encode(VI)) for EN, VI in pairs]

train_ids = numericalize_pairs(train_pairs, en_vocab, vi_vocab)
dev_ids   = numericalize_pairs(dev_pairs, en_vocab, vi_vocab)
test_ids  = numericalize_pairs(test_pairs, en_vocab, vi_vocab)

print("Train")
print("Tokens EN:", train_pairs[0][0])
print("IDs EN:   ", train_ids[0][0])
print("Tokens VI:", train_pairs[0][1])
print("IDs VI:   ", train_ids[0][1])
print()

print("Dev")
print("Tokens EN:", dev_pairs[0][0])
print("IDs EN:   ", dev_ids[0][0])
print("Tokens VI:", dev_pairs[0][1])
print("IDs VI:   ", dev_ids[0][1])
print()

print("Test")
print("Tokens EN:", test_pairs[0][0])
print("IDs EN:   ", test_ids[0][0])
print("Tokens VI:", test_pairs[0][1])
print("IDs VI:   ", test_ids[0][1])




Train
Tokens EN: ['rachel', 'pike', ':', 'the', 'science', 'behind', 'a', 'climate', 'headline']
IDs EN:    [1, 6284, 15935, 58, 6, 297, 555, 12, 684, 5279, 2]
Tokens VI: ['khoa học', 'đằng', 'sau', 'một', 'tiêu đề', 'về', 'khí hậu']
IDs VI:    [1, 293, 912, 84, 9, 2836, 33, 831, 2]

Dev
Tokens EN: ['this', 'is', 'not', 'a', 'finished', 'story', '.']
IDs EN:    [1, 21, 19, 43, 12, 1781, 196, 5, 2]
Tokens VI: ['câu chuyện', 'này', 'chưa', 'kết thúc', '.']
IDs VI:    [1, 185, 23, 168, 508, 5, 2]

Test
Tokens EN: ['when', 'i', 'was', 'little', ',', 'i', 'thought', 'my', 'country', 'was', 'the', 'best', 'on', 'the', 'planet', ',', 'and', 'i', 'grew', 'up', 'singing', 'a', 'song', 'called', '&quot', ';', 'nothing', 'to', 'envy', '.', '&quot', ';']
IDs EN:    [1, 59, 14, 26, 123, 4, 14, 185, 44, 255, 26, 6, 277, 31, 6, 480, 4, 8, 14, 986, 70, 1958, 12, 943, 161, 23, 7, 360, 9, 6796, 5, 23, 7, 2]
Tokens VI: ['khi', 'tôi', 'còn', 'nhỏ', ',', 'tôi', 'nghĩ', 'rằng', 'bắctriều tiên', 'là', 'đất n

## <a id='toc1_8_'></a>[Save the processed data](#toc0_)

Save vocabularies and numericalized train/dev/test pairs as pickle files in `data/processed/`.


In [9]:
OUT_DIR = 'D:/AIResearcher/MachineTraslation/data/processed'

with open(f'{OUT_DIR}/en_vocab.pkl', 'wb') as f:
    pickle.dump(en_vocab, f)
    
with open(f'{OUT_DIR}/vi_vocab.pkl', 'wb') as f:
    pickle.dump(vi_vocab, f)

with open(f'{OUT_DIR}/train_ids.pkl', 'wb') as f:
    pickle.dump(train_ids, f)

with open(f'{OUT_DIR}/dev_ids.pkl', 'wb') as f:
    pickle.dump(dev_ids, f)
    
with open(f'{OUT_DIR}/test_ids.pkl', 'wb') as f:
    pickle.dump(test_ids, f)
    

## <a id='toc1_9_'></a>[Verify the saved vocabulary](#toc0_)

Reload one saved vocabulary and compare its size with the in-memory version as a basic persistence check.


In [10]:
with open(f'{OUT_DIR}/en_vocab.pkl', 'rb') as f:
    check_vocab = pickle.load(f)
assert len(check_vocab) == len(en_vocab), "Vocab lÆ°u/Ä‘á»c láº¡i bá»‹ sai!"